# trailrunner in five minutes

One demand, carried end to end: **1000 kg of CO<sub>2</sub> captured from the air,
in Switzerland, in 2030.**

The usual way to answer that is to multiply a column of fixed coefficients. You
know what that looks like and you know where it stops: the column says the same
thing in Iceland as in Switzerland, in 2045 as in 2030, at one tonne as at a
million.

trailrunner answers it by *running* the supply chain instead. Every process is
Python code; the orchestrator asks one of them for the demand, gets back what
that process needs, and goes looking for whoever makes *that* — until there is
nothing left to ask. So this notebook is mostly about the machine: what the
pieces are, which piece holds which decision, and what the run leaves behind.

Seven beats. Nothing here touches the network: the vocabulary lookups come from
the committed `examples/pyst_cache.json`, the background datasets from the
committed `examples/background_pack.parquet`, and the parameters from the
committed parquet files beside this notebook.

In [1]:
import sys
from pathlib import Path

# Run from anywhere: nbconvert starts the kernel in the notebook's directory,
# a human might start it from the repository root.
EXAMPLES = Path.cwd() if (Path.cwd() / "showcase_models.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES))

from trailrunner import Demand, Flow
from trailrunner.models.dac import CO2_CAPTURED
from trailrunner.resolution import PystLabels

DEMAND = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2030), amount=1000.0, unit="kg"
)

# What a flow *is* is an IRI in https://vocab.sentier.dev -- not a free-text
# name. The vocabulary also knows what that concept is called, and those names
# are cached beside this notebook, so every print below reads in English with
# no network and no token.
VOCAB = PystLabels(EXAMPLES / "pyst_labels.json", client=None)


def name(iri: str, width: int | None = None) -> str:
    """The vocabulary's name for a concept, else the IRI's last segment."""
    label = VOCAB.label(iri) or iri.rsplit("/", 1)[-1]
    if width is not None and len(label) > width:
        label = label[: width - 1] + "\u2026"  # a column, not a claim: tree() prints it in full
    return label


print(DEMAND.amount, DEMAND.unit, DEMAND.flow.iri)
print("that IRI is:", name(DEMAND.flow.iri))
print("where:", DEMAND.flow.location, " when:", DEMAND.flow.time)

1000.0 kg https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_2811_21
that IRI is: Carbon dioxide
where: CH  when: 2030


## 1. The shape of the run

One demand goes in. Six objects pass it around until the queue is empty.

```mermaid
flowchart TB
    D([the demand]) --> Q[[Queue]]
    Q -->|pop| C{{ResolutionChain}}
    C -->|nobody offers| X[cutoff, with a reason]
    C -->|Offer: model + demand| R[Runner]
    R -->|apply| M[Model: your code]
    M -->|Result| R
    R -->|technosphere: what it needs| Q
    R -->|biosphere: what it emitted| I[(inventory)]
    X --> L[(Log)]
    R --> L
    L --> P([Report])
```

| Part | Its one job |
| --- | --- |
| `Demand` | an amount and a unit of a `Flow` — *what*, *where*, *when* |
| `Queue` | the demands still waiting; FIFO unless you hand it a priority |
| `ResolutionChain` | who can answer this demand? The first tier that offers wins |
| `Model` | one process, as code: `apply(demand) -> Result` |
| `Runner` | applies the model and validates the `Result` against the demand |
| `Log` | append-only: every node, edge, cutoff, fallback and rule |

**Every flow is keyed on an IRI from the [sentier vocabulary](https://vocab.sentier.dev).**
That is what lets two models written by two people meet at all: a `Demand` for
`.../BONSAI2025.1/fi_1730_9` finds whoever declared that IRI in `produces`, with
no name matching and no unit guessing anywhere in between. The vocabulary is
also *semantic and hierarchical* — a concept knows its own `skos:prefLabel`,
which is where the names printed below come from, and its `skos:broader` parent,
which is what beat 4 walks when nobody produces the exact concept asked for.

The seams are the point. The traversal never learns how a process works, and a
process never learns what else is in the supply chain: a model *returns*
demands rather than looking anything up, so it cannot reach into the graph and
does not know whether anyone will answer it. Everything that follows is those
six objects, one beat at a time.

## 2. A process is a function from `Demand` to `Result`

That is the whole model contract. `apply` receives the demand — the full amount,
never a unit demand — and answers three questions at once: what did I make, what
do I need, what did I emit?

In [2]:
import inspect

from trailrunner import LocationHierarchy, ParameterSet
from trailrunner.models import dac
from trailrunner.models.dac import DirectAirCapture

HIERARCHY = LocationHierarchy({"CH": "RER", "FR": "RER", "RER": "GLO"})
dac_params = ParameterSet.from_parquet(EXAMPLES / "dac_params.parquet", hierarchy=HIERARCHY)
plant = DirectAirCapture(params=dac_params)

answer = plant.apply(DEMAND)  # no orchestrator involved: a model is callable on its own

for field in ("production", "technosphere", "biosphere"):
    for exchange in getattr(answer, field):
        flow = exchange.flow
        print(
            f"{field:>13}  {exchange.amount:>8.1f} {exchange.unit:<4} "
            f"{name(flow.iri):<32} @{flow.location}/{flow.time}"
        )
print(f"{'provenance':>13}  {answer.provenance}")

   production    1000.0 kg   Carbon dioxide                   @CH/2030
 technosphere    5000.0 MJ   heat from main producers of heat @CH/2030
 technosphere     400.0 kWh  electricity                      @CH/2030
    biosphere   -1000.0 kg   co2-from-air                     @CH/2030
   provenance  {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


Three lists, three destinations, and the orchestrator needs to know nothing else
about direct air capture:

- `production` is checked against the demand that triggered the run and then
  dropped — it is an answer, not an input to anything;
- `technosphere` goes back on the queue, and is where the traversal comes from;
- `biosphere` is added to the inventory.

`provenance` is the model's own record of the parameter rows it read and the
fallbacks it took, and rides along to the report.

Why the demand is a function argument rather than a multiplier: the numbers in
`DirectAirCapture` depend on where and when it is asked. Colder, drier air
carries less CO<sub>2</sub> and less water to the sorbent per unit of air moved,
so the regeneration heat per kilogram goes up.

In [3]:
print(inspect.getsource(dac.ambient_penalty))

def ambient_penalty(temperature: float, humidity: float) -> float:
    """Multiplier on heat and electricity demand for non-reference air.

    Colder or drier than the reference gives a value above 1.0; warmer or
    wetter gives one below. Deliberately a simple linear response: the point is
    that the dependency exists and lives in code, not that this particular
    curve is the right one.
    """
    temperature_term = TEMPERATURE_SENSITIVITY * (REFERENCE_TEMPERATURE - temperature)
    humidity_term = HUMIDITY_SENSITIVITY * (REFERENCE_HUMIDITY - humidity)
    return 1.0 + temperature_term + humidity_term



In [4]:
print(f"{'where':>6} {'when':>6} {'degC':>6} {'RH':>6} {'penalty':>9} {'heat [MJ]':>11}")
for location in ("CH", "RER"):
    for year in (2020, 2030):
        air = dac_params.at(location=location, time=year)
        answer = plant.apply(
            Demand(flow=Flow(iri=CO2_CAPTURED, location=location, time=year),
                   amount=1000.0, unit="kg")
        )
        heat = [d for d in answer.technosphere if d.flow.iri == dac.HEAT][0]
        print(
            f"{location:>6} {year:>6} {air['temperature']:>6.1f} {air['humidity']:>6.2f} "
            f"{dac.ambient_penalty(air['temperature'], air['humidity']):>9.3f} "
            f"{heat.amount:>11.1f}"
        )

 where   when   degC     RH   penalty   heat [MJ]
    CH   2020    9.0   0.75     0.995      5970.0
    CH   2030   10.0   0.70     1.000      5000.0
   RER   2020   11.0   0.68     0.996      6573.6
   RER   2030   12.0   0.65     0.995      5472.5


Four answers to the same question, because the question was asked in four
different places and years. The line of `apply` that does it is

```python
heat = row["heat_demand"] * penalty * demand.amount
```

Nothing downstream rescales a `Result`, so a model whose response is *not*
proportional to the amount does not have to pretend it is. This one happens to
scale linearly; the ambient response is the part that could never have been a
coefficient.

**The process is the code.**

## 3. The loop

`Orchestrator.calculate` is a `while queue:` and little else. Pop a demand, ask
the chain who can answer it, hand the offer to the `Runner`, push the
`Result`'s technosphere demands back on, write everything to the `Log`.

Every seam in that sentence is an object you can replace, which also makes the
loop easy to watch: subclass the chain, print each demand it is asked about, and
the traversal narrates itself. Only the four shipped models are registered here.

In [5]:
from showcase_models import MODELS  # the same list `trailrunner run --models` loads
from trailrunner import Glossary, Orchestrator
from trailrunner.resolution import ModelProvider, ResolutionChain


class Narrating(ResolutionChain):
    """A chain that says what it was asked. The Orchestrator takes any chain."""

    def offer(self, demand, exclude=()):
        offer = super().offer(demand, exclude=exclude)
        who = type(offer.model).__name__ if offer else "cutoff (nobody offered)"
        print(f"pop {demand.amount:>9.4g} {demand.unit:<4} {name(demand.flow.iri, 32):<32} -> {who}")
        return offer


tier1 = ModelProvider(Glossary(MODELS))
first = Orchestrator(Narrating([tier1])).calculate(DEMAND)

pop      1000 kg   Carbon dioxide                   -> DirectAirCapture
pop      5000 MJ   heat from main producers of heat -> cutoff (nobody offered)
pop       400 kWh  electricity                      -> GridElectricity
pop     8.511 kWh  electricity-natural-gas          -> GasPower
pop      76.6 kWh  electricity-wind                 -> cutoff (nobody offered)
pop     340.4 kWh  electricity-hydro                -> cutoff (nobody offered)
pop     49.42 MJ   Natural gas, liquefied or in th… -> cutoff (nobody offered)


That is the whole traversal: seven pops, breadth-first, four of which nobody
could answer. Each `pop` after the first is a demand some earlier model returned.

The names come from the vocabulary, not from this notebook: `PystLabels` reads
each concept's `skos:prefLabel` out of a committed cache. Where a line still
shows a bare identifier — `electricity-wind`, `electricity-hydro` — there is no
label to read, because those IRIs are trailrunner's own invention rather than
vocabulary concepts. That is worth noticing: they are exactly the demands the
product dimension will not be able to generalise in beat 4.

Nothing was dropped and nothing was quietly zero. The four misses are cutoff
leaves in the report, each with a reason and a parent — and the `tree()` is the
log read back as the graph it recorded.

In [6]:
print(first.summary())
print()
print(first.tree(labels=VOCAB.label))  # the vocabulary's names, where it has one

3 nodes, 2 inventory entries
4 unresolved (no_model_found: 4)
0 proxies
attribution: allocation=none, capital=per_output

1000 kg Carbon dioxide @CH/2030  [model: DirectAirCapture]
  400 kWh electricity @CH/2030  [model: GridElectricity]
    8.51064 kWh electricity-natural-gas @CH/2030  [model: GasPower]
      49.4166 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: no_model_found]
    76.5957 kWh electricity-wind @CH/2030  [cutoff: no_model_found]
    340.426 kWh electricity-hydro @CH/2030  [cutoff: no_model_found]
  5000 MJ heat from main producers of heat @CH/2030  [cutoff: no_model_found]


Loops are bounded rather than solved: every visit is its own node, nodes are
never merged, and `max_depth` and `max_nodes` stop a cycle and set
`report.truncated`. A truncated tree with an honest cutoff list beats a
converged number nobody can check.

The same walk from the command line, no notebook involved:

```bash
uv run trailrunner run \
  "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_2811_21" \
  --amount 1000 --unit kg --location CH --year 2030 \
  --models examples/showcase_models.py
```

**Every node says how honestly it was answered.**

## 4. When nobody answers: the chain, tier by tier

`ResolutionChain` is a list of providers, asked in order, first offer wins. Tier
1 is the models. Every later tier is a concession — and the tier that made it
writes what it conceded into the node's resolution, so a proxy number is never
mistaken for an exact one. The order is yours to declare: no library default
decides whether a widened region beats a borrowed dataset.

**Tier 2 generalises the demand**, and this is what the vocabulary being
*hierarchical* buys. The heat demand is `fi_1730_9`, "heat from main producers
of heat". No model produces it. One level up `skos:broader` sits
`fi_1730`, "Steam and hot water" — a real BONSAI concept with a real parent
link, read here from the committed cache, with no network and no token. A gas
CHP registered at that parent can answer the demand once it is relaxed.

**Tier 3 borrows a dataset.** Give `DirectAirCapture` its `Fleet` and it demands
each running plant's construction **in the year that plant was built**. A
construction model turns that into steel and aluminium, and those come from the
curated background pack — ten real ecoinvent-derived datasets, all of them
`unit_process` rows.

**Two of the models below are written here, not shipped.** Nothing in this
repository produces `fi_1730`, and nothing in it co-produces — so there was no
target for the generalisation tier to find, and nothing for beat 5 to allocate.
`GasCHP` and `DacPlantConstruction` exist so those mechanisms have something to
bite on. Their efficiencies, prices and material intensities are invented.
Everything around them is not: the `skos:broader` walk, the pack lookup, the
completeness flag, the credit traversal and the construction pulse are the
library, and every block of output below is what it actually printed.

In [7]:
from trailrunner import AttributionSettings, Exchange, Fleet, Model, Property, Result, Settings
from trailrunner.models.dac import DAC_PLANT
from trailrunner.models.electricity import CO2_FOSSIL, ELECTRICITY, NATURAL_GAS

STEAM_AND_HOT_WATER = "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730"
STEEL = "https://vocab.sentier.dev/products/steel-low-alloyed"
ALUMINIUM = "https://vocab.sentier.dev/products/aluminium-primary"


class GasCHP(Model):
    """Gas-fired combined heat and power. Illustrative efficiencies and prices."""

    produces = [STEAM_AND_HOT_WATER]
    supports = frozenset({"economic", "substitution"})  # it co-produces; see beat 5

    heat_efficiency = 0.50
    electrical_efficiency = 0.35
    co2_per_mj_fuel = 0.056  # the same factor examples/gas_power_params.parquet carries
    heat_price = 0.02        # EUR/MJ
    electricity_price = 0.10  # EUR/kWh

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        fuel = demand.amount / self.heat_efficiency
        power = fuel * self.electrical_efficiency / 3.6
        return Result(
            production=[
                Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit,
                         properties=(Property("price", demand.amount * self.heat_price, "EUR"),)),
                Exchange(flow=Flow(iri=ELECTRICITY, **here), amount=power, unit="kWh",
                         properties=(Property("price", power * self.electricity_price, "EUR"),)),
            ],
            technosphere=[Demand(flow=Flow(iri=NATURAL_GAS, **here), amount=fuel, unit="MJ")],
            biosphere=[Exchange(flow=Flow(iri=CO2_FOSSIL, **here),
                                amount=fuel * self.co2_per_mj_fuel, unit="kg")],
            provenance={"fuel_mj": fuel},
        )


class DacPlantConstruction(Model):
    """What a capture plant is made of. Illustrative material intensities."""

    produces = [DAC_PLANT]
    supports = frozenset({"none", "economic", "substitution"})

    steel_per_capacity = 3.0      # kg steel per kg/year of capture capacity
    aluminium_per_capacity = 1.5  # kg aluminium, likewise

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        return Result(
            production=[Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit)],
            technosphere=[
                Demand(flow=Flow(iri=STEEL, **here),
                       amount=demand.amount * self.steel_per_capacity, unit="kg"),
                Demand(flow=Flow(iri=ALUMINIUM, **here),
                       amount=demand.amount * self.aluminium_per_capacity, unit="kg"),
            ],
            biosphere=[],
        )

In [8]:
from trailrunner.resolution import (
    BackgroundPack, BackgroundProvider, GeneralisingProvider, PystTaxonomy,
)

# The plants that were actually built: the fleet examples/dac.ipynb demonstrates.
FLEET_ROWS = [
    {"plant": "ch-pilot", "location": "CH", "build_year": 2007, "capacity": 5000.0, "lifetime": 20.0},
    {"plant": "ch-1", "location": "CH", "build_year": 2026, "capacity": 12000.0, "lifetime": 20.0},
    {"plant": "ch-2", "location": "CH", "build_year": 2029, "capacity": 40000.0, "lifetime": 20.0},
]
fleet = Fleet(FLEET_ROWS, units={"capacity": "kg/year", "lifetime": "year"}, hierarchy=HIERARCHY)

MODELS_PLUS = [
    DirectAirCapture(params=dac_params, fleet=fleet),
    *(model for model in MODELS if not isinstance(model, DirectAirCapture)),
    GasCHP(),
    DacPlantConstruction(),
]

tier1 = ModelProvider(Glossary(MODELS_PLUS))
# client=None: no network, ever. Every skos:broader answer comes from the file.
taxonomy = PystTaxonomy(EXAMPLES / "pyst_cache.json", client=None)
tier2 = GeneralisingProvider(tier1, hierarchy=HIERARCHY, taxonomy=taxonomy)
pack = BackgroundPack.from_parquet(EXAMPLES / "background_pack.parquet", hierarchy=HIERARCHY)
CHAIN = ResolutionChain([tier1, tier2, BackgroundProvider(pack)])


def walk(allocation):
    settings = Settings(attribution=AttributionSettings(allocation=allocation))
    return Orchestrator(CHAIN, settings=settings).calculate(DEMAND)


report = walk("economic")  # the CHP co-produces, so the run must state a rule: beat 5
print(report.summary())
print()
print(report.tree(labels=VOCAB.label))

10 nodes, 6 inventory entries
4 unresolved (generalisation_exhausted: 4)
5 proxies (4 incomplete)
attribution: allocation=economic, capital=per_output

1000 kg Carbon dioxide @CH/2030  [model: DirectAirCapture]
  5000 MJ heat from main producers of heat @CH/2030  [proxy: product: fi_1730_9 -> fi_1730]
    5070.42 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: generalisation_exhausted]
  400 kWh electricity @CH/2030  [model: GridElectricity]
    8.51064 kWh electricity-natural-gas @CH/2030  [model: GasPower]
      49.4166 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: generalisation_exhausted]
    76.5957 kWh electricity-wind @CH/2030  [cutoff: generalisation_exhausted]
    340.426 kWh electricity-hydro @CH/2030  [cutoff: generalisation_exhausted]
  11.5385 kg/year direct-air-capture-plant @CH/2026  [model: DacPlantConstruction]
    34.6154 kg steel-low-alloyed @CH/2026  [background: unit_process, incomplete]
    17.3077 kg aluminium-primary @

In [9]:
heat_node = [node for node in report.nodes if node.demand.flow.iri == dac.HEAT][0]
for key, value in report.proxies[heat_node.id].items():
    print(f"{key:>12}: {value}")

print()
print("   asked, in words:", name(dac.HEAT))
print("answered, in words:", name(STEAM_AND_HOT_WATER))

       model: GasCHP
 relaxations: ['product: fi_1730_9 -> fi_1730']
       asked: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730_9 @CH/2030
    answered: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730 @CH/2030
        tier: generalising

   asked, in words: heat from main producers of heat
answered, in words: Steam and hot water


The 5000 MJ cutoff is now a node tagged `[proxy: product: fi_1730_9 -> fi_1730]`,
and `report.proxies` says exactly what was asked and what was answered. The
steel and aluminium are tagged `[background: unit_process, incomplete]` — the
pack holds those datasets' *direct* exchanges only, so their own upstream is
missing, and the report says so rather than presenting a clean number.

Two things this chain deliberately does **not** do. The wind, hydro and
gas-share products are trailrunner's own invented IRIs, not vocabulary concepts,
so the product dimension cannot relax them and they stay cutoffs. And the
background pack has no electricity dataset on purpose: a grid-mix unit process
delegates its combustion upstream, so borrowing one would have answered a
kilowatt hour with a plausible-looking near-zero. A visible cutoff is better.

**We concede on purpose, along a declared hierarchy, and we log it.**

In [10]:
from trailrunner import viz
from trailrunner.assessment import Method, assess

# IPCC AR6 GWP100, stated here rather than read from a background database:
# the mapping from a gas to its warming potential is a fact about the gas.
GWP100 = Method(
    rows=[
        {"flow_iri": "https://vocab.sentier.dev/flows/co2-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 1.0},
        {"flow_iri": "https://vocab.sentier.dev/flows/ch4-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 29.8},
        {"flow_iri": "https://vocab.sentier.dev/flows/n2o", "flow_unit": "kg",
         "location": "GLO", "cf": 273.0},
        # The DAC model's uptake flow. The amount is already negative.
        {"flow_iri": dac.CO2_AIR, "flow_unit": "kg", "location": "GLO", "cf": 1.0},
    ],
    unit="kg CO2-eq",
    name="IPCC AR6 GWP100",
    hierarchy=HIERARCHY,
)

assessment = assess(report, GWP100)
sankey_figure = viz.sankey(report, assessment=assessment)
sankey_figure

## 5. Where a value judgement enters the loop

The CHP makes heat *and* electricity. How its burden is split between them is
not a measurement; it is a choice, and different defensible choices give
different answers. trailrunner will not make it for you — and the place it
refuses is a specific one: the `Runner`, between applying the model and
validating what came back. The model neither makes the choice nor sees it, which
is why the rule lands in `report.attribution` and not in the model's provenance.

In [11]:
from trailrunner import UnallocatedCoProduction

try:
    walk("none")
except UnallocatedCoProduction as refusal:
    print("allocation='none' ->", refusal)

allocation='none' -> GasCHP returned co-products (https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100) but the run's allocation rule is 'none'; model it monofunctionally or choose a rule


In [12]:
runs = {rule: walk(rule) for rule in ("economic", "substitution")}

for rule, run in runs.items():
    score = assess(run, GWP100).score
    print(f"{rule:>13}: {score:>9.1f} kg CO2-eq   ({run.summary().splitlines()[1]})")

credited = runs["substitution"]
chp_node = [node for node in credited.nodes if node.demand.flow.iri == dac.HEAT][0]
print()
print("what the CHP node recorded under substitution:")
for key, value in credited.attribution[chp_node.id].items():
    print(f"  {key}: {value}")

     economic:    -580.6 kg CO2-eq   (4 unresolved (generalisation_exhausted: 4))
 substitution:    -311.3 kg CO2-eq   (7 unresolved (generalisation_exhausted: 7, of which 3 on a credit branch))

what the CHP node recorded under substitution:
  allocation: substitution
  share: 1.0
  substituted: ['https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100']


The same model, the same 1000 kg: removed CO<sub>2</sub> on one rule, and
roughly half that on the other. Under `economic` the CHP's fuel and
emissions are partitioned by revenue; under `substitution` the heat carries all
of them and is credited with the Swiss grid electricity it displaces — which is
mostly hydro, so the credit is small. That credit is traversed as a *negative*
demand, answered by someone other than the CHP, and `summary()` counts its
cutoffs separately, because a forgone credit overstates an impact where a
forgone burden understates it.

The gap between the two is set by `GasCHP`'s invented heat and electricity
prices — revenue is what `economic` partitions on. What is demonstrated is that
the rule moves the answer and that the report records which rule ran, not that
either number is right for a real CHP.

**The rule is on the report, next to the number it produced.**

## 6. Time rides along

Nothing in the loop ever had to be told about time: a `Flow` carries its year
the way it carries its location, so every demand pushed, every emission
accumulated and every node logged is already dated. The construction of `ch-1`
is emitted in 2026 and `ch-2` in 2029; the capture and the heat that drives it
are in 2030.

So the inventory *is* a time series, and can be characterized as one — IPCC AR6
impulse-response functions, year by year. No matrix rebuilt, no second model,
and no step in the traversal that knew this was coming.

In [13]:
from trailrunner.assessment import assess_dynamic

# No characterization table is passed: default_functions() maps the DAC uptake
# flow to the ordinary CO2 function, because the model emits it as an already
# negative CO2 exchange and nothing should negate it a second time.
dynamic = assess_dynamic(report, metric="radiative_forcing", horizon=100)
print(dynamic.summary())

by_year = dynamic.series.groupby(dynamic.series["date"].dt.year)["amount"].sum()
print()
print("marginal radiative forcing, first years [W/m2]:")
print(by_year.head(6).to_string())

-5.14244e-11 W·yr/m2
metric: radiative_forcing, horizon: 100 years
horizon anchored at: 2026-01-01
0 uncharacterized exchanges
0 wrong unit exchanges
0 undated exchanges
0 beyond-horizon exchanges
4 unresolved
5 proxies

marginal radiative forcing, first years [W/m2]:
date
2027    5.054518e-14
2028    9.235309e-14
2029    4.282174e-14
2030    1.684839e-13
2031   -9.756895e-13
2032   -1.776486e-12


In [14]:
curve_figure = viz.curve(dynamic)
curve_figure

The bars before 2030 are the plants being built. The dive after it is the
capture paying that back, and the cumulative line crosses zero in the year it
does because of *when* each kilogram happened, not only how much of it there was.

A static score cannot say any of this. It gives one number for a project whose
warming and whose cooling are four years apart.

The pulse's size is `DacPlantConstruction`'s illustrative intensities; the
shape — warming first, cooling later — is what the traversal produced from the
dates it carried.

**`bw_temporalis` and `bw_timex` get here too — from a matrix. This got here
because the traversal never lost the date.**

## 7. The record

The `Report` is a reading of the `Log`, not a replacement for it: the log itself
goes to one parquet file under one schema, so two runs can be diffed with a
single read.

In [15]:
import pyarrow.parquet as pq

print(report.summary())

out = Path("showcase_log.parquet")
report.log.to_parquet(out)
table = pq.read_table(out)
print()
print(f"wrote {out.name}: {table.num_rows} rows, {table.num_columns} columns")
print("kinds:", sorted(set(table.column("kind").to_pylist())))
out.unlink()

10 nodes, 6 inventory entries
4 unresolved (generalisation_exhausted: 4)
5 proxies (4 incomplete)
attribution: allocation=economic, capital=per_output

wrote showcase_log.parquet: 142 rows, 19 columns
kinds: ['attribution', 'biosphere', 'node', 'provenance', 'resolution', 'unresolved']


In [16]:
contributions_figure = viz.contributions(
    assessment, by="node", labels={node.id: node.model for node in report.nodes}
)
contributions_figure

Parameters in as parquet, the whole run out as parquet: every node, every
cutoff, every parameter fallback, every proxy and the rule that produced the
number. This notebook read four parameter files, one background pack and one
vocabulary cache, all committed, and asked the network for nothing.

**Parquet in, parquet out, every choice on the record.**

---

- The installation page: [`content/installation.md`](../content/installation.md)
- The deeper worked example: [`examples/dac.ipynb`](dac.ipynb)